In [1]:
# Cell 1 — Imports
import pyroll.core as pr
import pyroll.wusatowski_spreading
import matplotlib.pyplot as plt
import numpy as np

print("Imports ✓")

Imports ✓


In [2]:
# Cell 2 — What is a hook?
# A hook is a function slot that PyRolL calls during solving
# Multiple functions can register to the same hook
# PyRolL calls them in order until one returns a non-None value

# Let's see all hooks available on RollPass
roll_pass_hooks = [
    attr for attr in dir(pr.RollPass)
    if isinstance(getattr(pr.RollPass, attr), pr.Hook)
]

print(f"Total hooks on RollPass: {len(roll_pass_hooks)}")
print()
print("First 20 hooks:")
for h in sorted(roll_pass_hooks)[:20]:
    print(f"  {h}")

Total hooks on RollPass: 60

First 20 hooks:
  abs_draught
  abs_elongation
  abs_spread
  back_tension
  contact_area
  contact_friction
  contact_pressure
  deformation_resistance
  disk_element_count
  displaced_cross_section
  draught
  duration
  elongation
  elongation_efficiency
  energy_consumption
  entry_point
  exit_point
  forward_slip_ratio
  free_surface_area
  front_tension


In [5]:
# Cell 3 — Hook anatomy

# A hook works like this:
#
# 1. PyRolL defines the hook slot:
#    RollPass.gap = Hook[float]("gap", "Roll gap in metres")
#
# 2. Plugins register functions to that slot:
#    @RollPass.gap
#    def my_gap_function(self: RollPass):
#        return some_calculation(self)
#
# 3. During solving PyRolL calls all registered functions
#    in order until one returns a non-None value
#
# 4. You can OVERRIDE by registering your own function
#    that returns a different value

# Let's see what's currently registered on the gap hook
# Cell 3 — Hook anatomy fixed
print("Functions registered on RollPass.gap hook:")
for f in pr.RollPass.gap.functions:
    print(f"  {f.function.__module__}.{f.function.__name__}")

print()
print("Functions registered on RollPass.roll_force hook:")
for f in pr.RollPass.roll_force.functions:
    print(f"  {f.function.__module__}.{f.function.__name__}")

print()
print("Functions registered on RollPass.spread hook:")
for f in pr.RollPass.spread.functions:
    print(f"  {f.function.__module__}.{f.function.__name__}")

Functions registered on RollPass.gap hook:
  pyroll.core.roll_pass.hookimpls.two_roll_pass.gap

Functions registered on RollPass.roll_force hook:
  pyroll.core.roll_pass.hookimpls.symmetric_roll_pass.roll_force

Functions registered on RollPass.spread hook:
  pyroll.core.roll_pass.hookimpls.deformation_unit.spread


In [6]:
# Check spread hook before and after wusatowski
print("Spread functions BEFORE wusatowski already loaded:")
for f in pr.RollPass.spread.functions:
    print(f"  {f.function.__module__}.{f.function.__name__}")

# Now check — wusatowski was already imported at top
# so let's check what it added
import pyroll.wusatowski_spreading
print()
print("Spread functions AFTER wusatowski import:")
for f in pr.RollPass.spread.functions:
    print(f"  {f.function.__module__}.{f.function.__name__}")

Spread functions BEFORE wusatowski already loaded:
  pyroll.core.roll_pass.hookimpls.deformation_unit.spread

Spread functions AFTER wusatowski import:
  pyroll.core.roll_pass.hookimpls.deformation_unit.spread


In [8]:
# Cell 4 — Your first custom hook

# Define in_profile and sequence first
in_profile = pr.Profile.round(
    radius=30e-3,
    temperature=1200 + 273.15,
    strain=0,
    material=["steel", "C15"],
    flow_stress=100e6,
    density=7.5e3,
    specific_heat_capacity=690,
    thermal_conductivity=23,
    environment_temperature=20 + 273.15,
    surface_heat_transfer_coefficient=100,
)

sequence = pr.PassSequence([
    pr.RollPass(
        label="R1 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=50e-3, depth=10e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=8e-3,
    ),
    pr.Transport(label="R1->R2", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R2 - Round",
        roll=pr.Roll(
            groove=pr.FalseRoundGroove(r1=2e-3, r2=25e-3, depth=12e-3, flank_angle=30),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=12e-3,
    ),
    pr.Transport(label="R2->R3", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R3 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=30e-3, depth=6e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
    pr.Transport(label="R3->R4", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R4 - Round",
        roll=pr.Roll(
            groove=pr.RoundGroove(r1=1e-3, r2=35e-3, depth=4e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
])

sequence.solve(in_profile)

# Check original force values
print("BEFORE custom hook:")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

# Register custom hook
@pr.RollPass.roll_force
def my_custom_force(self: pr.RollPass):
    area = self.roll.contact_area
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    return flow_stress * area * 1.5

# Solve again with custom hook active
print()
print("AFTER custom hook registered:")
sequence.solve(in_profile)
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

BEFORE custom hook:
  R1 - Oval: 0.445 MN
  R2 - Round: 0.251 MN
  R3 - Oval: 0.230 MN
  R4 - Round: 0.206 MN

AFTER custom hook registered:
  R1 - Oval: 0.667 MN
  R2 - Round: 0.377 MN
  R3 - Oval: 0.344 MN
  R4 - Round: 0.310 MN


In [9]:
# Cell 4 — Your first custom hook
# Let's override roll_force with a custom calculation
# and verify PyRolL uses our value

# First check current roll_force values
print("BEFORE custom hook:")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

# Now register a custom hook that scales force by 1.5x
# to verify our hook is being called
@pr.RollPass.roll_force
def my_custom_force(self: pr.RollPass):
    # Get contact area
    area = self.roll.contact_area
    # Get mean flow stress
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    # Simple force = flow_stress × contact_area × 1.5 (our custom factor)
    return flow_stress * area * 1.5

print()
print("AFTER custom hook registered:")
sequence.solve(in_profile)
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

BEFORE custom hook:
  R1 - Oval: 0.667 MN
  R2 - Round: 0.377 MN
  R3 - Oval: 0.344 MN
  R4 - Round: 0.310 MN

AFTER custom hook registered:
  R1 - Oval: 0.667 MN
  R2 - Round: 0.377 MN
  R3 - Oval: 0.344 MN
  R4 - Round: 0.310 MN


In [10]:
# Cell 5 — Prepend hook to run first

# Remove previous hook first
pr.RollPass.roll_force.functions.remove(my_custom_force)

# Register with prepend=True so it runs BEFORE core hook
@pr.RollPass.roll_force.prepend
def my_custom_force_v2(self: pr.RollPass):
    area = self.roll.contact_area
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    return flow_stress * area * 1.5

print("Custom hook registered with prepend ✓")
print()

sequence.solve(in_profile)

print("AFTER prepend hook:")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

AttributeError: 'Hook' object has no attribute 'prepend'

In [11]:
# Check available methods on Hook
print([m for m in dir(pr.RollPass.roll_force) if not m.startswith('_')])

['add_function', 'functions', 'functions_gen', 'get_result', 'name', 'owner', 'remove_function', 'type']


In [12]:
# Cell 5 — Register hook at position 0 (runs first)

# First remove previous hook
pr.RollPass.roll_force.remove_function(my_custom_force)

# Define new function
def my_custom_force_v2(self: pr.RollPass):
    area = self.roll.contact_area
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    return flow_stress * area * 1.5

# Check add_function signature
import inspect
print(inspect.signature(pr.RollPass.roll_force.add_function))

(func, tryfirst=False, trylast=False, wrapper=False)


In [13]:
# Cell 6 — Register hook with tryfirst=True

def my_custom_force_v2(self: pr.RollPass):
    area = self.roll.contact_area
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    return flow_stress * area * 1.5

pr.RollPass.roll_force.add_function(my_custom_force_v2, tryfirst=True)

print("Custom hook registered with tryfirst=True ✓")
print()

sequence.solve(in_profile)

print("AFTER custom hook (tryfirst):")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

print()
print("Original values were:")
print("  R1: 0.667 MN")
print("  R2: 0.377 MN")
print("  R3: 0.344 MN")
print("  R4: 0.310 MN")

Custom hook registered with tryfirst=True ✓

AFTER custom hook (tryfirst):
  R1 - Oval: 0.667 MN
  R2 - Round: 0.377 MN
  R3 - Oval: 0.344 MN
  R4 - Round: 0.310 MN

Original values were:
  R1: 0.667 MN
  R2: 0.377 MN
  R3: 0.344 MN
  R4: 0.310 MN


In [14]:
# Cell 7 — Verify hook is called

pr.RollPass.roll_force.remove_function(my_custom_force_v2)

def my_custom_force_v3(self: pr.RollPass):
    print(f"  → my hook called for {self.label}")
    area = self.roll.contact_area
    flow_stress = (self.in_profile.flow_stress + self.out_profile.flow_stress) / 2
    result = flow_stress * area * 2.0  # 2x multiplier — should be very different
    print(f"  → returning {result/1e6:.3f} MN")
    return result

pr.RollPass.roll_force.add_function(my_custom_force_v3, tryfirst=True)

print("Solving with debug hook:")
sequence.solve(in_profile)

print()
print("Final forces:")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.3f} MN")

AttributeError: 'function' object has no attribute 'function'

In [15]:
# Cell 7 — Fresh kernel approach
# Don't remove — just define and add with tryfirst

def my_debug_hook(self: pr.RollPass):
    print(f"  → hook called for {self.label}")
    return 99e6  # return obvious value 99 MN so we know it's our hook

pr.RollPass.roll_force.add_function(my_debug_hook, tryfirst=True)

print("Solving:")
sequence.solve(in_profile)

print()
print("Forces (should be 99 MN if our hook runs):")
for unit in sequence:
    if isinstance(unit, pr.RollPass):
        print(f"  {unit.label}: {unit.roll_force/1e6:.1f} MN")
        

Solving:
  → hook called for R1 - Oval
  → hook called for R1 - Oval
  → hook called for R1 - Oval
  → hook called for R1 - Oval
  → hook called for R1 - Oval
  → hook called for R1 - Oval
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R2 - Round
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R3 - Oval
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook called for R4 - Round
  → hook call

In [17]:
# Cell 8 — Hitchcock roll flattening hook
# Hitchcock equation: R' = R × (1 + C_H × F / (b × Δh))
# where:
#   R' = flattened roll radius
#   R  = nominal roll radius
#   C_H = 16(1-ν²)/(πE) — Hitchcock constant for steel rolls
#   F  = rolling force
#   b  = strip width
#   Δh = draft (height reduction)

def hitchcock_flattened_radius(self: pr.RollPass):
    """Calculate elastically flattened roll radius using Hitchcock equation"""
    # Material constants for steel rolls
    E_roll = 210e9    # Young's modulus of roll (Pa)
    nu_roll = 0.3     # Poisson's ratio of roll

    # Hitchcock constant
    C_H = 16 * (1 - nu_roll**2) / (np.pi * E_roll)

    # Rolling parameters
    R = self.roll.nominal_radius          # nominal roll radius
    F = self.roll_force                   # rolling force
    b = self.in_profile.width             # strip width
    dh = self.in_profile.height - self.gap  # draft

    if dh <= 0:
        return R  # no deformation → no flattening

    # Hitchcock equation
    R_prime = R * (1 + C_H * F / (b * dh))

    return R_prime

# Add to RollPass as a new hook result — print comparison
pr.RollPass.roll_force.remove_function(my_debug_hook)

# Solve with original force first
sequence.solve(in_profile)

print("Hitchcock Roll Flattening Analysis:")
print("="*60)
print(f"{'Pass':<15} {'R nominal':>12} {'R flattened':>12} {'Δ (%)':>8}")
print("="*60)

for unit in sequence:
    if isinstance(unit, pr.RollPass):
        R_nom = unit.roll.nominal_radius * 1000
        R_flat = hitchcock_flattened_radius(unit) * 1000
        delta = (R_flat - R_nom) / R_nom * 100
        print(f"{unit.label:<15} {R_nom:>12.1f} {R_flat:>12.1f} {delta:>8.2f}%")

print("="*60)
print("R nominal  = original roll radius (mm)")
print("R flattened = elastically deformed radius under load (mm)")

AttributeError: 'function' object has no attribute 'function'

In [1]:
# CELL 1 — Fresh start, all in one cell
import pyroll.core as pr
import pyroll.wusatowski_spreading
import pyroll.integral_thermal
import matplotlib.pyplot as plt
import numpy as np

in_profile = pr.Profile.round(
    radius=30e-3,
    temperature=1200 + 273.15,
    strain=0,
    material=["steel", "C15"],
    flow_stress=100e6,
    density=7.5e3,
    specific_heat_capacity=690,
    thermal_conductivity=23,
    environment_temperature=20 + 273.15,
    surface_heat_transfer_coefficient=100,
)

sequence = pr.PassSequence([
    pr.RollPass(
        label="R1 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=50e-3, depth=10e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=8e-3,
    ),
    pr.Transport(label="R1->R2", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R2 - Round",
        roll=pr.Roll(
            groove=pr.FalseRoundGroove(r1=2e-3, r2=25e-3, depth=12e-3, flank_angle=30),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=12e-3,
    ),
    pr.Transport(label="R2->R3", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R3 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=30e-3, depth=6e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
    pr.Transport(label="R3->R4", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R4 - Round",
        roll=pr.Roll(
            groove=pr.RoundGroove(r1=1e-3, r2=35e-3, depth=4e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
])

sequence.solve(in_profile)
print("Solved ✓")

Solved ✓


In [2]:
# CELL 2 — Hitchcock analysis (no hook manipulation needed)
def hitchcock_flattened_radius(roll_pass):
    E_roll = 210e9
    nu_roll = 0.3
    C_H = 16 * (1 - nu_roll**2) / (np.pi * E_roll)

    R  = roll_pass.roll.nominal_radius
    F  = roll_pass.roll_force
    b  = roll_pass.in_profile.width
    dh = roll_pass.in_profile.height - roll_pass.gap

    if dh <= 0:
        return R

    R_prime = R * (1 + C_H * F / (b * dh))
    return R_prime

print("Hitchcock Roll Flattening Analysis:")
print("="*60)
print(f"{'Pass':<15} {'R nom (mm)':>12} {'R flat (mm)':>12} {'Δ (%)':>8}")
print("="*60)

for unit in sequence:
    if isinstance(unit, pr.RollPass):
        R_nom  = unit.roll.nominal_radius * 1000
        R_flat = hitchcock_flattened_radius(unit) * 1000
        delta  = (R_flat - R_nom) / R_nom * 100
        print(f"{unit.label:<15} {R_nom:>12.1f} {R_flat:>12.1f} {delta:>8.2f}%")

print("="*60)

Hitchcock Roll Flattening Analysis:
Pass              R nom (mm)  R flat (mm)    Δ (%)
R1 - Oval              155.0        155.5     0.31%
R2 - Round             155.0        155.5     0.32%
R3 - Oval              155.0        155.7     0.44%
R4 - Round             155.0        155.8     0.53%


In [3]:
# Cell 3 — Why foil rolling is different
print("Flattening significance comparison:")
print()

cases = [
    ("Thick bar (our sim)", 60e-3,  0.445e6, 155e-3),
    ("Medium bar",          20e-3,  0.300e6, 155e-3),
    ("Thin strip (1mm)",    1e-3,   0.200e6, 155e-3),
    ("Foil (0.1mm)",        0.1e-3, 0.150e6, 155e-3),
]

E_roll = 210e9
nu_roll = 0.3
C_H = 16 * (1 - nu_roll**2) / (np.pi * E_roll)
b = 100e-3  # 100mm width for all cases

print(f"{'Case':<25} {'h (mm)':>8} {'R nom':>8} {'R flat':>8} {'Δ (%)':>8} {'Δ/h (%)':>10}")
print("="*75)

for name, h, F, R in cases:
    dh = h * 0.3  # 30% reduction
    R_prime = R * (1 + C_H * F / (b * dh))
    delta = (R_prime - R) * 1000
    delta_pct = (R_prime - R) / R * 100
    delta_vs_h = delta / (h * 1000) * 100
    print(f"{name:<25} {h*1000:>8.2f} {R*1000:>8.1f} {R_prime*1000:>8.2f} "
          f"{delta_pct:>8.3f}% {delta_vs_h:>10.1f}%")

print("="*75)
print()
print("Δ/h = flattening as % of strip thickness")
print("When Δ/h >> 1% → roll deformation dominates → must model it!")

Flattening significance comparison:

Case                        h (mm)    R nom   R flat    Δ (%)    Δ/h (%)
Thick bar (our sim)          60.00    155.0   155.85    0.546%        1.4%
Medium bar                   20.00    155.0   156.71    1.103%        8.6%
Thin strip (1mm)              1.00    155.0   177.81   14.713%     2280.5%
Foil (0.1mm)                  0.10    155.0   326.04  110.347%   171038.5%

Δ/h = flattening as % of strip thickness
When Δ/h >> 1% → roll deformation dominates → must model it!
